In [ ]:
import dolphindb as ddb
import pandas as pd

# 连接
s = ddb.session()
s.connect("******", "******", "******", "******")

# 1. 本地读取沪深300成分股
hs300_df = pd.read_csv(r"../数据/沪深300成分股.csv")

# 2. 上传到 DolphinDB
s.upload({"hs300_components": hs300_df})

# 3. 正确的查询语句
df = s.run("""
// 加载数据库
db = database(""******"");
stock_daily = loadTable(db, "stock_daily_data");
stock_factor_daily=loadTable(db,"stock_factor_daily")//total_mv
stock_info_table=loadTable(db,"stock_info_table")//industry
// 取出股票代码列表
code_list = exec con_code from hs300_components;

// 查询
t = select
    a.*,
    b.total_mv as market_cap,
    c.industry
from stock_daily as a
left join stock_factor_daily as b
on a.ts_code = b.ts_code and a.trade_date = b.trade_date
left join stock_info_table as c
on a.ts_code = c.ts_code
where
    a.ts_code in code_list
    and a.trade_date >= 2016.03.01
    and a.trade_date <= 2026.03.01
    and a.close > 0
    and a.vol > 0
    and a.pct_chg >= -10.05 and  a.pct_chg<=10.05  // 剔除涨跌停
    and not isNull(b.total_mv)        // 剔除市值缺失
    and not isNull(c.industry);          // 剔除行业缺失
           
select * from t;
""")

# 保存
df.to_csv(r"../数据/沪深300成分股_10年.csv", index=False)
print("完成！数据量：", len(df))

完成！数据量： 641110
